# Assignment No. 3
## Generate Public/Private Keys and Simulate a Digital Wallet

**Objective:** Use ECDSA (secp256k1) cryptography to generate a digital wallet with a
public/private key pair, derive a wallet address, and sign/verify messages.

### What We Will Learn
- Asymmetric cryptography: public vs private keys
- ECDSA (Elliptic Curve Digital Signature Algorithm) - used by Bitcoin and Ethereum
- How a wallet address is derived from a public key
- Digital signatures: sign with private key, verify with public key


## 1. Core Concepts

### Key Pair Cryptography
- **Private Key:** A secret random number. Never share this! Used to SIGN transactions.
- **Public Key:** Derived from private key mathematically. Share freely. Used to VERIFY signatures.
- **Wallet Address:** A shortened hash of the public key. Your public identity on the blockchain.

### ECDSA on secp256k1
Bitcoin and Ethereum both use the `secp256k1` elliptic curve.
- Private key: 256-bit random number
- Public key: a point (x, y) on the elliptic curve
- One-way: you can compute public key from private, but NOT vice versa

### Address Derivation
```
Private Key --> Public Key --> SHA-256 --> RIPEMD-160 --> Wallet Address
```


## 2. Install Required Library


In [1]:
import sys
!{sys.executable} -m pip install cryptography --quiet
print('cryptography library ready.')

cryptography library ready.



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Import Libraries


In [2]:
import hashlib
import os
import binascii

from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.asymmetric.utils import (
    decode_dss_signature, encode_dss_signature
)
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.backends import default_backend

print('All libraries imported successfully!')

All libraries imported successfully!


## 4. Key Generation


In [3]:
def generate_key_pair():
    """
    Generate an ECDSA key pair on the secp256k1 curve.
    Returns (private_key_object, public_key_object)
    """
    private_key = ec.generate_private_key(
        ec.SECP256K1(),
        default_backend()
    )
    public_key = private_key.public_key()
    return private_key, public_key


def get_private_key_hex(private_key):
    """Return the private key as a hex string (like Bitcoin WIF concept)."""
    private_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    # Extract the 32-byte raw scalar from DER
    raw_int = private_key.private_numbers().private_value
    return format(raw_int, '064x')  # 64 hex chars = 32 bytes


def get_public_key_hex(public_key):
    """Return the uncompressed public key as hex (04 || x || y)."""
    pub_bytes = public_key.public_bytes(
        encoding=serialization.Encoding.X962,
        format=serialization.PublicFormat.UncompressedPoint
    )
    return pub_bytes.hex()


print('Key generation functions defined.')

Key generation functions defined.


## 5. Wallet Address Derivation

Bitcoin uses: `SHA-256(public_key)` then `RIPEMD-160` to get a 20-byte address.
We implement the same pipeline.


In [4]:
def public_key_to_address(public_key):
    """
    Derive a wallet address from a public key.
    Pipeline: PublicKey -> SHA-256 -> RIPEMD-160 -> hex address
    """
    # Step 1: Get raw public key bytes (uncompressed)
    pub_bytes = public_key.public_bytes(
        encoding=serialization.Encoding.X962,
        format=serialization.PublicFormat.UncompressedPoint
    )

    # Step 2: SHA-256 hash
    sha256_hash = hashlib.sha256(pub_bytes).digest()

    # Step 3: RIPEMD-160 hash
    ripemd160 = hashlib.new('ripemd160')
    ripemd160.update(sha256_hash)
    address_bytes = ripemd160.digest()

    return address_bytes.hex()  # 40 hex chars = 20 bytes


print('Address derivation function defined.')

Address derivation function defined.


## 6. Create a Wallet


In [5]:
class Wallet:
    def __init__(self, name='Anonymous'):
        self.name        = name
        self._priv_key, self._pub_key = generate_key_pair()
        self.address     = public_key_to_address(self._pub_key)
        self.private_hex = get_private_key_hex(self._priv_key)
        self.public_hex  = get_public_key_hex(self._pub_key)

    def sign(self, message: str) -> bytes:
        """Sign a message string with the private key."""
        msg_bytes = message.encode('utf-8')
        signature = self._priv_key.sign(msg_bytes, ec.ECDSA(hashes.SHA256()))
        return signature

    def display(self):
        print(f'  Wallet Owner : {self.name}')
        print(f'  Address      : {self.address}')
        print(f'  Public Key   : {self.public_hex[:32]}...')
        print(f'  Private Key  : {self.private_hex[:16]}... (NEVER share!)')


def verify_signature(public_key_obj, message: str, signature: bytes) -> bool:
    """Verify a signature using the signer's public key."""
    try:
        msg_bytes = message.encode('utf-8')
        public_key_obj.verify(signature, msg_bytes, ec.ECDSA(hashes.SHA256()))
        return True
    except Exception:
        return False


print('Wallet class defined.')

Wallet class defined.


## 7. Demo: Generate Two Wallets


In [6]:
print('Creating wallets for Alice and Bob...')
print()
alice = Wallet('Alice')
bob   = Wallet('Bob')

print('=== ALICE WALLET ===')
alice.display()
print()
print('=== BOB WALLET ===')
bob.display()

Creating wallets for Alice and Bob...

=== ALICE WALLET ===
  Wallet Owner : Alice
  Address      : 39d0d4151dd4ee13c4173d63380f8b23352aedc1
  Public Key   : 045f348f165bae689510976719d02712...
  Private Key  : 6197e8f262ea60a6... (NEVER share!)

=== BOB WALLET ===
  Wallet Owner : Bob
  Address      : cc6b7daa5e8de42db83871c9b46ea17a8c070862
  Public Key   : 04188720ef987114a1dc58e0cc3444e1...
  Private Key  : 4a62d99518c08e1c... (NEVER share!)


## 8. Sign and Verify a Message

Alice signs a transaction message. Bob (or anyone) can verify it was Alice who signed.


In [7]:
message = 'Alice sends 10 BTC to Bob'
print(f'Message: "{message}"')
print()

# Alice signs the message
signature = alice.sign(message)
print(f'Signature (hex): {signature.hex()[:48]}...')
print(f'Signature length: {len(signature)} bytes')
print()

# Verify: correct key should pass
valid = verify_signature(alice._pub_key, message, signature)
print(f'Verify with Alice public key  : {valid}')

# Verify: wrong key should fail
invalid = verify_signature(bob._pub_key, message, signature)
print(f'Verify with Bob public key    : {invalid}')

# Verify: tampered message should fail
tampered = verify_signature(alice._pub_key, message + ' TAMPERED', signature)
print(f'Verify with tampered message  : {tampered}')

Message: "Alice sends 10 BTC to Bob"

Signature (hex): 3046022100f3377d6e630d2f9f144df7b6c44d66ca51b8d0...
Signature length: 72 bytes

Verify with Alice public key  : True
Verify with Bob public key    : False
Verify with tampered message  : False


## 9. Understanding Key Formats


In [8]:
print('KEY FORMAT DETAILS')
print('=' * 60)
print(f'Private key (hex, 32 bytes = 256 bits):')
print(f'  {alice.private_hex}')
print()
print(f'Public key (uncompressed, 65 bytes):')
print(f'  Prefix 04 = uncompressed point')
print(f'  X coord  : {alice.public_hex[2:66]}')
print(f'  Y coord  : {alice.public_hex[66:]}')
print()
print(f'Wallet Address (20 bytes = 40 hex chars):')
print(f'  {alice.address}')
print()
print('Address derivation path:')
print('  Public Key -> SHA-256 -> RIPEMD-160 -> Address')
print('=' * 60)

KEY FORMAT DETAILS
Private key (hex, 32 bytes = 256 bits):
  6197e8f262ea60a6b14d161d8113ef19ce33cf6506181d57ca34144b914ab299

Public key (uncompressed, 65 bytes):
  Prefix 04 = uncompressed point
  X coord  : 5f348f165bae689510976719d02712579fdd09daa59fa719fde0a53ffe3e8ff4
  Y coord  : 25137b2a37a475c3af6de66cb7d876bef0560246efb6ebbd469d4cefda4deb0a

Wallet Address (20 bytes = 40 hex chars):
  39d0d4151dd4ee13c4173d63380f8b23352aedc1

Address derivation path:
  Public Key -> SHA-256 -> RIPEMD-160 -> Address


## 10. Conclusion

### Summary
| Component | Algorithm | Size |
|---|---|---|
| Private Key | Random number on secp256k1 | 256 bits / 32 bytes |
| Public Key | EC point derived from private | 65 bytes (uncompressed) |
| Wallet Address | SHA-256 + RIPEMD-160 of public key | 20 bytes / 40 hex chars |
| Signature | ECDSA with SHA-256 | ~70-72 bytes |

**Security Guarantee:**
- Only the owner of the private key can create a valid signature
- Anyone with the public key can verify the signature
- Tampering with the signed message invalidates the signature
- The private key cannot be derived from the public key or address
